# xarray & zarr Pipeline

This notebook demonstrates the full cloud-native data pipeline:

1. Fetch multi-station waveforms → xarray Dataset
2. Manipulate with xarray (select, resample, compute)
3. Write to zarr store
4. Read back and verify
5. Pattern for earth2studio / GPU pipeline interop

**Requires:** `pip install seisfetch[xarray,zarr]`

In [ ]:
import numpy as np
from pathlib import Path

from seisfetch import (
    SeisfetchClient,
    parse_mseed,
    bundle_to_xarray,
    to_zarr,
    TraceBundle,
)

## 1. Fetch → xarray Dataset

`get_xarray()` does: S3 download → pymseed decode → xarray.Dataset
in one call.  Each channel becomes a DataArray with a `datetime64[ns]`
time coordinate.

In [ ]:
client = SeisfetchClient(backend="s3_open")

# Fetch 10 minutes of IU.ANMO broadband
ds = client.get_xarray(
    "IU", "ANMO",
    starttime="2024-01-15T00:00:00",
    endtime="2024-01-15T00:10:00",
)
print(ds)

## 2. Inspect the Dataset

In [ ]:
for var_name in ds.data_vars:
    da = ds[var_name]
    print(f"\n{var_name}:")
    print(f"  shape:         {da.shape}")
    print(f"  dtype:         {da.dtype}")
    print(f"  time range:    {str(da.time.values[0])[:26]} → {str(da.time.values[-1])[:26]}")
    print(f"  sampling_rate: {da.attrs['sampling_rate']} Hz")
    print(f"  network:       {da.attrs['network']}")
    print(f"  station:       {da.attrs['station']}")
    print(f"  channel:       {da.attrs['channel']}")
    print(f"  min/max:       {da.values.min()} / {da.values.max()}")

## 3. Build multi-station Datasets

Fetch several stations and combine into one Dataset.
Each station-channel gets its own DataArray.

In [ ]:
# Fetch from multiple stations
stations = [
    ("IU", "ANMO", "00", "BHZ"),
    ("IU", "CCM",  "00", "BHZ"),
]

all_traces = []
for net, sta, loc, cha in stations:
    try:
        bundle = client.get_numpy(
            net, sta, starttime="2024-01-15T00:00:00",
            endtime="2024-01-15T00:10:00",
            location=loc, channel=cha,
        )
        all_traces.extend(bundle.traces)
        print(f"  {net}.{sta}.{loc}.{cha}: {len(bundle)} traces")
    except Exception as e:
        print(f"  {net}.{sta}.{loc}.{cha}: FAILED — {e}")

# Combine into one Dataset
combined = TraceBundle(all_traces)
ds_multi = bundle_to_xarray(combined)
print(f"\nCombined Dataset: {len(ds_multi.data_vars)} channels")
print(ds_multi)

## 4. xarray operations

Standard xarray operations work on the Dataset.

In [ ]:
# Select a single channel
if "IU_ANMO_00_BHZ" in ds_multi.data_vars:
    da = ds_multi["IU_ANMO_00_BHZ"]
    print(f"IU.ANMO.00.BHZ: {da.shape}")

    # Time slicing
    da_slice = da.sel(time=slice("2024-01-15T00:02:00", "2024-01-15T00:05:00"))
    print(f"  Sliced to 3 min: {da_slice.shape}")

    # Basic stats
    print(f"  Mean: {float(da.mean()):.1f}")
    print(f"  Std:  {float(da.std()):.1f}")

## 5. Write to zarr

In [ ]:
zarr_path = "multi_station.zarr"
to_zarr(ds_multi, zarr_path)
print(f"Wrote {zarr_path}")

# List what's in the store
for p in sorted(Path(zarr_path).iterdir()):
    if p.is_dir():
        size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
        print(f"  {p.name}/  ({size/1e3:.1f} KB)")

## 6. Read back from zarr

In [ ]:
import xarray as xr

ds_loaded = xr.open_zarr(zarr_path)
print(ds_loaded)

# Verify data integrity
for var in ds_multi.data_vars:
    if var in ds_loaded.data_vars:
        orig = ds_multi[var].values
        loaded = ds_loaded[var].values
        match = np.array_equal(orig, loaded)
        print(f"  {var}: {'✓ match' if match else '✗ MISMATCH'}")

## 7. earth2studio interop pattern

NVIDIA earth2studio expects data as xarray Datasets or torch tensors.
The pipeline from `seisfetch` to earth2studio:

```python
# 1. Fetch seismic data → xarray
ds = client.get_xarray("IU", "ANMO", channel="BHZ",
                        starttime="2024-01-15", endtime="2024-01-16")

# 2. Save to zarr (cloud-native, S3-compatible)
to_zarr(ds, "s3://my-bucket/seismic.zarr")

# 3. In earth2studio, load directly:
# ds = xr.open_zarr("s3://my-bucket/seismic.zarr")

# 4. Convert to torch tensor for GPU processing:
import torch
data = torch.from_numpy(ds["IU_ANMO_00_BHZ"].values).float()
data = data.unsqueeze(0).unsqueeze(0)  # (batch, channel, time)
```

The zarr format is natively supported by:
- xarray (`open_zarr`)
- dask (lazy/chunked loading)
- fsspec (S3, GCS, Azure blob)
- earth2studio data sources

In [ ]:
# Quick demo: xarray → numpy → simulate GPU pipeline
if "IU_ANMO_00_BHZ" in ds_loaded.data_vars:
    data_np = ds_loaded["IU_ANMO_00_BHZ"].values.astype(np.float32)
    print(f"numpy array: shape={data_np.shape}, dtype={data_np.dtype}")

    # This is what you'd feed to torch:
    # data_torch = torch.from_numpy(data_np).unsqueeze(0).unsqueeze(0)
    # print(f"torch tensor: {data_torch.shape}")  # (1, 1, N)

## 8. Append mode: growing a zarr store over time

For long-running mining campaigns, append new data to an existing store:

```python
# Day 1
ds1 = client.get_xarray("IU", "ANMO", channel="BHZ",
                         starttime="2024-01-15", endtime="2024-01-16")
to_zarr(ds1, "campaign.zarr", mode="w")

# Day 2 — append along time dimension
ds2 = client.get_xarray("IU", "ANMO", channel="BHZ",
                         starttime="2024-01-16", endtime="2024-01-17")
to_zarr(ds2, "campaign.zarr", mode="a", append_dim="time")
```

## Cleanup

In [ ]:
import shutil
shutil.rmtree("multi_station.zarr", ignore_errors=True)
print("Cleaned up.")

## Summary

| Step | Code | Output |
|------|------|--------|
| Fetch + parse | `client.get_xarray(...)` | `xarray.Dataset` |
| Multi-station | `TraceBundle` → `bundle_to_xarray()` | merged Dataset |
| Save | `to_zarr(ds, "path.zarr")` | zarr directory store |
| Load | `xr.open_zarr("path.zarr")` | lazy Dataset |
| GPU | `torch.from_numpy(ds[var].values)` | torch Tensor |